# 2D Nonlocal Low-Rank Denoiser (k-NN + SVD / NNM / WNNM) — CPU-only

**Goal:** Denoise 2D images with repeatable structure under heavy noise by exploiting **self-similarity**.  
For each anchor patch:
1) Find `K` similar patches in a local window (k-NN).  
2) Stack them into a `P×K` matrix (`P = a²`, `a = patch size`).  
3) Apply low-rank shrinkage (SVD soft-threshold / NNM / WNNM).  
4) Rebuild patches and aggregate them back with smooth overlap weights.

**Why it works:** Structure is shared by similar patches → low rank; noise is spread across many directions → suppressed by shrinking small singular values.

**This notebook uses CPU SVD only** (NumPy/MKL/OpenBLAS). No CuPy/Torch changes required.  
For large images, start with faster params, then tighten for final quality.


In [1]:
from __future__ import annotations

# Core
import numpy as np
import matplotlib.pyplot as plt

# Imaging
from skimage import io, img_as_float32, color, exposure, util, data
from skimage.util import view_as_windows
from skimage.restoration import estimate_sigma as sk_estimate_sigma
from scipy.ndimage import gaussian_laplace

# UX
from tqdm.auto import tqdm
import ipywidgets as W

# Defaults (WSL-friendly)
DATA_DIR = "/home/askiran/data/"

# (Optional) let BLAS use your 16 cores
try:
    from threadpoolctl import threadpool_limits
    threadpool_limits(16)
    print("BLAS threads set to 16")
except Exception:
    pass


BLAS threads set to 16


## Configuration (edit-friendly)

**Parameters**
- `patch` *(int)*: Patch width `a` (pixels). Common: 6–10. Default **8**.
- `search_radius` *(int)*: k-NN half-window `r` (window is `(2r+1)²`). Common: 12–24. Default **16**.
- `K` *(int)*: Number of similar patches per group. Common: 16–32. Default **24**.
- `anchor_step` *(int)*: Stride of anchor locations (top-left of patches). Larger → faster. Default **4**.
- `method` *(str)*: `"svd_soft"`, `"nnm"`, or `"wnnm"`.
- `tau` *(float | None)*: Soft threshold for SVD/NNM. If `None`, auto set from noise `sigma`.
- `wnnm_c` *(float)*: Weight scale for WNNM shrinkage (typ. 0.5–2.0). Default **1.0**.
- `sigma` *(float | None)*: Noise std in `[0,1]`. If `None`, estimate robustly.
- `mean_normalize` *(bool)*: Subtract per-patch mean for distance & denoise (recommended).
- `kaiser_beta` *(float)*: Kaiser window softness for aggregation. Default **2.0**.
- `dtype` *(np.dtype)*: Internal accumulators dtype. Default **np.float32**.

**Auto-tau:** If `tau is None`, we use `tau = sigma * sqrt(2 * max(P, K))`, with `P=a²`.

**Tuning cheatsheet**
- **Too smooth / detail loss:** ↓ `tau`, or use `method='wnnm'` with smaller `wnnm_c`, or reduce `K`.
- **Residual noise:** ↑ `K` or `search_radius`.
- **Slow runtime:** ↑ `anchor_step`, ↓ `search_radius` or `K`; crop ROI while tuning.


In [2]:
# ---- Denoiser configuration ----
cfg = {
    "patch": 8,              # a
    "search_radius": 16,     # r
    "K": 32,                 # K similar patches
    "anchor_step": 4,
    "method": "svd_soft",    # 'svd_soft' | 'nnm' | 'wnnm'
    "tau": None,             # None => auto from sigma
    "wnnm_c": 1.0,
    "sigma": None,           # None => estimate
    "mean_normalize": True,
    "kaiser_beta": 2.0,
    "dtype": np.float32,
}
print("cfg set ✅")


cfg set ✅


## Utilities

- `estimate_sigma(img)`: Noise std in `[0,1]`. First tries `skimage` estimator; fallback is MAD on a LoG high-pass.
- `kaiser2d(a, beta)`: Smooth 2D window for overlap aggregation; reduces seams.

**Possible pitfalls**
- If noise is not i.i.d. Gaussian, sigma estimation may be biased (≈40%). Manually set `cfg["sigma"]` if needed.
- `kaiser_beta` too small can leave seams; too large can slightly blur overlaps.


In [3]:
def estimate_sigma(img: np.ndarray, fallback_log_sigma: float = 1.2) -> float:
    """Estimate noise std for a 2D image in [0,1]."""
    try:
        s = float(sk_estimate_sigma(img, average_sigmas=True, channel_axis=None))
        if np.isfinite(s) and s > 0:
            return s
    except Exception:
        pass
    hp = gaussian_laplace(img, sigma=fallback_log_sigma)
    mad = np.median(np.abs(hp - np.median(hp)))
    return float(1.4826 * mad)

def kaiser2d(a: int, beta: float = 2.0, dtype=np.float32) -> np.ndarray:
    w1d = np.kaiser(a, beta).astype(dtype)
    w2d = np.outer(w1d, w1d)
    w2d /= (w2d.max() + 1e-12)
    return w2d


## k-NN patch grouping helpers

- `extract_window_patches(img, y, x, a, r)`: All `a×a` patches within a window centered at `(y, x)` with radius `r`. Returns a lazy view `(Ny, Nx, a, a)` and top-left coordinates.
- `topk_similar(view, anchor_patch, K, mean_normalize)`: Mean-normalized L2 distance (if enabled) to find K closest patches.

**Common failure points**
- **Wrong neighbors (~70%)**: If patch means vary a lot (lighting, bias), enable `mean_normalize=True`.
- **K too small (~50%)**: Weak low-rank structure; increase `K` to ~24–32.


In [4]:
def extract_window_patches(img: np.ndarray, y: int, x: int, a: int, r: int):
    """Extract all a×a patches inside a window centered at (y,x) with radius r.
    Returns:
        view: (Ny, Nx, a, a)
        top_lefts: list of (yy, xx) top-left coords per patch in row-major order
    """
    H, W = img.shape
    y0 = max(0, y - r)
    y1 = min(H - a, y + r)
    x0 = max(0, x - r)
    x1 = min(W - a, x + r)

    region = img[y0:y1 + a, x0:x1 + a]  # include patch size
    view = view_as_windows(region, (a, a))  # (Ny, Nx, a, a)
    Ny, Nx = view.shape[:2]
    top_lefts = [(y0 + iy, x0 + ix) for iy in range(Ny) for ix in range(Nx)]
    return view, top_lefts

def topk_similar(view: np.ndarray, anchor_patch: np.ndarray, K: int, mean_normalize: bool = True):
    """Return sorted indices of K nearest patches to anchor by (mean-normalized) L2."""
    Ny, Nx, a, b = view.shape
    assert a == b
    P = a * a
    V = view.reshape(Ny * Nx, P).astype(np.float32, copy=False)
    A = anchor_patch.reshape(1, P).astype(np.float32)

    if mean_normalize:
        V_mean = V.mean(axis=1, keepdims=True)
        A_mean = A.mean(axis=1, keepdims=True)
        Vn = V - V_mean
        An = A - A_mean
    else:
        Vn, An = V, A

    d2 = np.sum((Vn - An)**2, axis=1)
    K_eff = min(K, d2.size)
    idx = np.argpartition(d2, K_eff - 1)[:K_eff]
    idx = idx[np.argsort(d2[idx])]
    return idx


## Low-rank shrinkage (CPU SVD)

`svd_shrink_cpu(M, method, tau, wnnm_c, sigma)` denoises a `P×K` group:

- **SVD/Nuclear Norm (NNM)**: `s_i ← max(s_i − τ, 0)` (soft-threshold).
- **WNNM** (weighted NNM): stronger shrink on smaller singular values:
  \[
  s_i \leftarrow \max\left(s_i - \frac{c\cdot 2\sigma^2}{s_i + \varepsilon},\, 0\right)
  \]

**Tips**
- Start with `method='svd_soft'`. If texture gets too smooth, try `wnnm` with `wnnm_c≈0.7`.


In [5]:
def svd_shrink_cpu(M: np.ndarray, method: str, tau: float, wnnm_c: float, sigma: float) -> np.ndarray:
    """Low-rank denoising via SVD shrinkage on CPU. M shape: (P, K) float32."""
    U, S, VT = np.linalg.svd(M, full_matrices=False)

    if method in ("svd_soft", "nnm"):
        S_shr = np.maximum(S - tau, 0.0, dtype=np.float32)
    elif method == "wnnm":
        T = (wnnm_c * 2.0 * (sigma**2)) / (S + 1e-8)
        S_shr = np.maximum(S - T.astype(np.float32), 0.0, dtype=np.float32)
    else:
        raise ValueError(f"Unknown method: {method}")

    Mhat = (U * S_shr) @ VT
    return Mhat.astype(np.float32, copy=False)


## The denoiser

`denoise_image(img, cfg)`:
- Convert to grayscale float32 `[0,1]`.
- Estimate `sigma` if needed; set `tau` if `None`.
- Precompute Kaiser window.
- For each anchor:
  1) extract local window patches,
  2) select `K` nearest (mean-normalized),
  3) form `P×K` matrix, center if `mean_normalize`,
  4) low-rank shrinkage (CPU SVD),
  5) re-add per-patch means (if centered) and aggregate with weights.
- Normalize by the sum of weights.

**Complexity controls:** `anchor_step`, `search_radius`, `K`.


In [6]:
def svd_shrink_cpu(M: np.ndarray, method: str, tau: float, wnnm_c: float, sigma: float) -> np.ndarray:
    """Low-rank denoising via SVD shrinkage on CPU. M shape: (P, K) float32."""
    U, S, VT = np.linalg.svd(M, full_matrices=False)

    if method in ("svd_soft", "nnm"):
        S_shr = np.maximum(S - tau, 0.0, dtype=np.float32)
    elif method == "wnnm":
        T = (wnnm_c * 2.0 * (sigma**2)) / (S + 1e-8)
        S_shr = np.maximum(S - T.astype(np.float32), 0.0, dtype=np.float32)
    else:
        raise ValueError(f"Unknown method: {method}")

    Mhat = (U * S_shr) @ VT
    return Mhat.astype(np.float32, copy=False)


## Main function: `denoise_image(img, cfg)`

**Steps per anchor (top-left strides by `anchor_step`):**
1. Extract local window patches and top-left coords.
2. k-NN: select `K` nearest to anchor (mean-normalized L2 if enabled).
3. Build `P×K` matrix; optionally subtract per-patch mean.
4. Low-rank shrinkage (SVD soft / NNM / WNNM) on CPU.
5. Re-add per-patch means (if subtracted).
6. Aggregate patches into `num/den` using a **Kaiser** window.
7. Normalize (`out = num/den`) and clip to `[0,1]`.

**Complexity controls:** `anchor_step`, `search_radius`, `K`.  
**Numerics:** Use `float32` to keep memory down.


In [7]:
def denoise_image(img_in: np.ndarray, cfg: dict) -> tuple[np.ndarray, dict]:
    a = int(cfg["patch"])
    r = int(cfg["search_radius"])
    K = int(cfg["K"])
    step = int(cfg["anchor_step"])
    method = str(cfg["method"]).lower()
    tau = cfg["tau"]
    wnnm_c = float(cfg["wnnm_c"])
    sigma = cfg["sigma"]
    mean_norm = bool(cfg["mean_normalize"])
    beta = float(cfg["kaiser_beta"])
    dtype = cfg["dtype"]

    # Prepare image as grayscale float32 [0,1]
    img = img_in
    if img.ndim == 3:
        img = color.rgb2gray(img)
    img = img_as_float32(img)
    img = np.clip(img, 0.0, 1.0)

    H, W = img.shape
    if sigma is None:
        sigma = estimate_sigma(img)

    P = a * a
    if tau is None:
        tau = float(sigma * np.sqrt(2.0 * max(P, K)))

    wpatch = kaiser2d(a, beta=beta, dtype=dtype)

    # Accumulators
    num = np.zeros((H, W), dtype=dtype)
    den = np.zeros((H, W), dtype=dtype)

    ys = range(0, H - a + 1, step)
    xs = range(0, W - a + 1, step)

    for y in tqdm(ys, total=len(range(0, H - a + 1, step)), desc="Rows"):
        for x in xs:
            anchor = img[y:y+a, x:x+a].astype(np.float32, copy=False)

            view, top_lefts = extract_window_patches(img, y, x, a, r)
            Ny, Nx = view.shape[:2]
            V = view.reshape(Ny * Nx, a*a)

            idx = topk_similar(view, anchor, K, mean_normalize=mean_norm)
            if idx.size == 0:
                continue

            group = V[idx].T  # (P, K)

            # mean-center per patch if requested
            if mean_norm:
                patch_means = group.mean(axis=0, keepdims=True)
                group_c = group - patch_means
            else:
                patch_means = None
                group_c = group

            # low-rank shrinkage (CPU SVD)
            Mhat = svd_shrink_cpu(group_c.astype(np.float32, copy=False),
                                  method, float(tau), float(wnnm_c), float(sigma))

            if mean_norm:
                Mhat = Mhat + patch_means

            # aggregate
            for k_local, flat_idx in enumerate(idx):
                yy, xx = top_lefts[flat_idx]
                patch_hat = Mhat[:, k_local].reshape(a, a)
                num[yy:yy+a, xx:xx+a] += patch_hat * wpatch
                den[yy:yy+a, xx:xx+a] += wpatch

    out = np.zeros_like(img)
    mask = den > 0
    out[mask] = num[mask] / den[mask]
    out[~mask] = img[~mask]

    out = np.clip(out, 0.0, 1.0)
    meta = {"sigma": float(sigma), "tau": float(tau), "method": method}
    return out.astype(dtype, copy=False), meta


## Robust image loading (WSL paths, TIFF/OME, bit-depths, multi-page)

- Accept **Windows paths** (auto-convert to WSL `/mnt/<drive>/...`).
- Prefer `tifffile` for TIFF/OME; otherwise `imageio` / `skimage`.
- Convert to **float32 in `[0,1]`** and **grayscale**.
- Print diagnostics (shape, dtype, min/max).


In [8]:
import os, glob, pathlib
import imageio.v3 as iio
from typing import Optional, Tuple

def win_to_wsl_path(p: str) -> str:
    """Map 'C:\\path\\file.tif' → '/mnt/c/path/file.tif' on WSL."""
    if not isinstance(p, str) or len(p) < 3:
        return p
    if p[1:3] in (":\\", ":/"):
        drive = p[0].lower()
        rest = p[2:].replace("\\", "/")
        return f"/mnt/{drive}/{rest.lstrip('/')}"
    return p

def print_diag(name: str, arr: np.ndarray):
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, "
          f"min={float(arr.min()):.6f}, max={float(arr.max()):.6f}")

def _normalize_to_float01(arr: np.ndarray) -> np.ndarray:
    if np.issubdtype(arr.dtype, np.floating):
        out = arr.astype(np.float32, copy=False)
        if out.min() < 0 or out.max() > 1.0:  # rescale if not already [0,1]
            mn, mx = float(out.min()), float(out.max())
            if mx > mn:
                out = (out - mn) / (mx - mn)
        return np.clip(out, 0.0, 1.0)
    elif np.issubdtype(arr.dtype, np.integer):
        info = np.iinfo(arr.dtype)
        return (arr.astype(np.float32) / info.max).clip(0, 1)
    return img_as_float32(arr)

def _to_gray(img: np.ndarray) -> np.ndarray:
    if img.ndim == 2:
        return img
    if img.ndim == 3:
        if img.shape[2] == 1:
            return img[..., 0]
        if img.shape[2] in (3, 4):
            try:
                return color.rgb2gray(img[..., :3])
            except Exception:
                return img[..., :3].mean(axis=2)
        return img.mean(axis=2)
    raise ValueError(f"Cannot grayscale shape {img.shape}")

def load_image_any(path: str, page: Optional[int] = None) -> Tuple[np.ndarray, dict]:
    """
    Load robustly; return (gray_float01, info).
    info: {'raw_shape','raw_dtype','pages','used_page'}
    """
    p = win_to_wsl_path(path)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Path not found: {p}")

    ext = pathlib.Path(p).suffix.lower()
    raw = None
    used_page = None
    pages = 1

    if ext in {".tif", ".tiff"} or ".ome.tif" in p.lower() or ".ome.tiff" in p.lower():
        try:
            import tifffile as tiff
            with tiff.TiffFile(p) as tf:
                n_pages = len(tf.pages)
                if n_pages > 1:
                    used_page = 0 if page is None else int(np.clip(page, 0, n_pages-1))
                    raw = tf.pages[used_page].asarray()
                    pages = n_pages
                else:
                    raw = tf.asarray()
        except Exception as e:
            print(f"[warn] tifffile failed ({str(e).splitlines()[0]}), trying imageio")
            raw = iio.imread(p)
            if raw.ndim >= 3 and raw.shape[0] > 4:
                pages = raw.shape[0]
    else:
        try:
            raw = iio.imread(p)
            if raw.ndim >= 3 and raw.shape[0] > 4:
                pages = raw.shape[0]
        except Exception:
            raw = io.imread(p)
            pages = 1

    info = {"raw_shape": tuple(raw.shape), "raw_dtype": str(raw.dtype), "pages": int(pages), "used_page": used_page}

    # If a stack came back (Z,H,W) or (Z,H,W,C)
    arr = raw
    if arr.ndim == 3 and pages > 1 and arr.shape[0] == pages:
        used_page = 0 if page is None else int(np.clip(page, 0, pages-1))
        arr = arr[used_page]
        info["used_page"] = used_page
    elif arr.ndim == 4 and pages > 1:
        used_page = 0 if page is None else int(np.clip(page, 0, arr.shape[0]-1))
        arr = arr[used_page]
        info["used_page"] = used_page

    arr = _normalize_to_float01(arr)
    arr = _to_gray(arr)
    arr = np.ascontiguousarray(arr, dtype=np.float32)

    print_diag("Loaded image", arr)
    if pages > 1:
        print(f"Stack detected: pages={pages}, using page={info['used_page']}")
    return arr, info


## Point to your file and load

- Accepts **Windows** or **WSL** paths.
- If `img_path` is `None`, we’ll list examples in `DATA_DIR`.


In [9]:
# Example: r"C:\Users\you\Pictures\sample.tif"  -> auto-mapped
# Example: "/mnt/c/Users/you/Pictures/sample.tif"
# Example: f"{DATA_DIR}your_image.tif"
img_path = "/home/askiran/data/Peri_1/ANF+PVA2%,4ml_SCAN 3times_Cross_17_crop.tif"  # <-- set this to your file path

if img_path is None:
    patterns = ("*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg", "*.bmp")
    found = []
    for pat in patterns:
        found.extend(glob.glob(os.path.join(DATA_DIR, pat)))
    found = sorted(found)[:10]
    if found:
        print("Examples in DATA_DIR:")
        for f in found:
            print(" -", f)
    else:
        print("No sample files found in DATA_DIR; set img_path above and re-run.")
else:
    try:
        img0, info0 = load_image_any(img_path, page=None)
        noisy = img0
        used_noisy = False
    except Exception as e:
        print("Load failed:", e)
        img0 = None


Loaded image: shape=(1013, 1511), dtype=float32, min=0.000000, max=1.000000


## If the file is a multi-page TIFF/OME, pick a page (optional)

If your file is 2D, this section is a no-op.


In [11]:
from IPython.display import display

if (img_path is not None) and (("tif" in str(img_path).lower()) or ("tiff" in str(img_path).lower())) and ('img0' not in globals() or img0 is None):
    try:
        import tifffile as tiff
        p = win_to_wsl_path(img_path)
        with tiff.TiffFile(p) as tf:
            n_pages = len(tf.pages)
        if n_pages > 1:
            @W.interact(page=W.IntSlider(0, 0, n_pages-1, 1))
            def _pick(page=0):
                img_page, info_page = load_image_any(img_path, page=page)
                globals()["img0"] = img_page
                globals()["noisy"] = img_page
                print(f"Selected page: {page}")
        else:
            print("Single-page TIFF.")
    except Exception as e:
        print("Page selection skipped (not a TIFF or tifffile not available):", e)


## Interactive denoising UI **with Save**

**What this does**
- Lets you tweak `K`, `τ (tau)`, `search_radius`, `method`, and `anchor_step`.
- Runs the denoiser on your current `noisy` image.
- Keeps the **last result in memory** and lets you **save** it with one click.

**UI controls**
- **K** *(int)*: # of similar patches (typ. 16–32). Higher = cleaner but slower.
- **auto τ** *(bool)*: If on, τ is computed from estimated σ and P=a².
- **τ** *(float)*: Soft threshold; lower preserves detail, higher removes more noise.
- **search_radius** *(int)*: Half-window for k-NN; larger finds better matches but costs time.
- **method** *(str)*: `'svd_soft'`, `'nnm'` (same as soft), `'wnnm'` (weighted shrink).
- **anchor_step** *(int)*: Anchor stride; larger is faster with slightly fewer samples.
- **Output path** *(text)*: Where to save. Use `.tif/.tiff` to keep **float32** (recommended), or `.png/.jpg` (saved as 8-bit).
- **Preserve float32 (TIFF)** *(checkbox)*: If true and path ends with `.tif[f]`, saves as float32 via `tifffile` when available.

**Notes & pitfalls**
- Saving PNG/JPEG will quantize to **8-bit** (`uint8`). Use **TIFF** for quantitative work.
- If the folder doesn’t exist, it will be created automatically.
- If `tifffile` isn’t installed, TIFF saves fall back to `skimage.io.imsave` (still OK for float→8/16-bit).
- Common errors: bad path/permissions (~40%), unsupported extension (~20%).


In [10]:
import os, time
from datetime import datetime
from skimage import io as skio, util as skutil

# ---- State to hold the last output ----
_last_out = {"img": None, "meta": None}

# ---- Widgets ----
K_w             = W.IntSlider(value=cfg["K"], min=8, max=48, step=1, description="K")
tau_auto_w      = W.Checkbox(value=True, description="auto τ")
tau_w           = W.FloatLogSlider(value=0.02, base=10, min=-3, max=0, step=0.05, description="τ")
search_r_w      = W.IntSlider(value=cfg["search_radius"], min=8, max=32, step=1, description="search_radius")
method_w        = W.Dropdown(options=["svd_soft", "nnm", "wnnm"], value=cfg["method"], description="method")
anchor_step_w   = W.IntSlider(value=cfg["anchor_step"], min=2, max=12, step=1, description="anchor_step")

# Default save name in your data dir
default_name = os.path.join(
    DATA_DIR, f"denoised_{datetime.now().strftime('%Y%m%d_%H%M%S')}.tif"
)
out_path_w        = W.Text(value=default_name, description="Output path", layout=W.Layout(width="75%"))
preserve_f32_w    = W.Checkbox(value=True, description="Preserve float32 (TIFF)")
run_btn           = W.Button(description="Denoise", button_style="primary")
save_btn          = W.Button(description="Save result", button_style="success")
status_out        = W.Output()

# ---- Denoise action ----
def _run_denoise(_=None):
    with status_out:
        status_out.clear_output()
        # Build local cfg
        local = cfg.copy()
        local["K"]             = int(K_w.value)
        local["search_radius"] = int(search_r_w.value)
        local["method"]        = method_w.value
        local["anchor_step"]   = int(anchor_step_w.value)
        local["tau"]           = None if tau_auto_w.value else float(tau_w.value)

        # Run
        t0 = time.time()
        out, meta = denoise_image(noisy, local)
        dt = time.time() - t0

        # Stash result
        _last_out["img"]  = out
        _last_out["meta"] = meta

        # Show
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1); plt.imshow(noisy, cmap="gray"); plt.title("Input"); plt.axis("off")
        plt.subplot(1,2,2); plt.imshow(out,   cmap="gray"); plt.title(f"Denoised ({local['method']})"); plt.axis("off")
        plt.show()

        print(f"σ̂: {meta['sigma']:.4f}   τ: {meta['tau']:.4f}   method: {meta['method']}   time: {dt:.2f}s")

# ---- Save action ----
def _save_result(_=None):
    with status_out:
        # Basic checks
        if _last_out["img"] is None:
            print("No result to save — run Denoise first.")
            return

        path = out_path_w.value.strip()
        if not path:
            print("Please provide an output path.")
            return

        # Ensure folder exists
        folder = os.path.dirname(path) or "."
        try:
            os.makedirs(folder, exist_ok=True)
        except Exception as e:
            print("Could not create output directory:", e)
            return

        img = _last_out["img"]
        ext = os.path.splitext(path)[1].lower()

        try:
            if ext in (".tif", ".tiff"):
                # TIFF preferred for float32
                if preserve_f32_w.value:
                    try:
                        import tifffile as tiff
                        tiff.imwrite(path, img.astype(np.float32, copy=False))
                    except Exception as e:
                        print("[warn] tifffile not available or failed; saving as 16-bit via skimage:", e)
                        skio.imsave(path, skutil.img_as_uint(img))  # 16-bit
                else:
                    # 16-bit integer TIFF
                    skio.imsave(path, skutil.img_as_uint(img))
            elif ext in (".png",):
                # PNG requires integer; 8-bit is common
                skio.imsave(path, skutil.img_as_ubyte(img))
            elif ext in (".jpg", ".jpeg", ".bmp"):
                skio.imsave(path, skutil.img_as_ubyte(img))
            else:
                # Default safe fallback: 8-bit PNG with appended extension if none
                if ext == "":
                    path = path + ".png"
                skio.imsave(path, skutil.img_as_ubyte(img))

            print("Saved:", path)
        except Exception as e:
            print("Save failed:", e)

# Hook up buttons
run_btn.on_click(_run_denoise)
save_btn.on_click(_save_result)

# Layout
row1 = W.HBox([K_w, search_r_w, anchor_step_w])
row2 = W.HBox([tau_auto_w, tau_w, method_w])
row3 = W.HBox([out_path_w])
row4 = W.HBox([preserve_f32_w, run_btn, save_btn])

ui = W.VBox([row1, row2, row3, row4, status_out])
display(ui)

# If τ is auto, gray out the manual τ slider for clarity (optional nicety)
def _toggle_tau(change):
    tau_w.disabled = change["new"]
tau_auto_w.observe(_toggle_tau, names="value")
tau_w.disabled = tau_auto_w.value

print("Interactive UI ready. Set parameters → Denoise → Save.")


Interactive UI ready. Set parameters → Denoise → Save.


# Parameter Tuning Playbook — 2D Nonlocal Low-Rank Denoiser (k-NN + SVD/NNM/WNNM)

Use this as a checklist to optimize parameters and choose methods for **your** images.

---

## Before you tune (2 quick setup steps)

1. **Pick a small ROI** (e.g., 256×256) that contains key textures + edges. Tune on the ROI; reuse those settings on the full image.
2. **Inspect 3 views during tuning**
   - Denoised image
   - **Residual** = noisy − denoised (should look like grain, not structure)
   - **Zoom 200–400%** on textured area to catch oversmoothing and seams

**Tell-tales**
- Residual shows **edge/texture ghosts** → τ too high (over-shrink)
- Output still **speckly** while residual is weak → τ too low or K too small

---

## Parameter-by-parameter tuning

### 1) `patch` (a) — patch size
- **Start:** 8  
- **Increase to 9–10** if texture is **coarse/periodic** and you want stronger collaborative averaging  
- **Decrease to 6–7** if tiny features/edges disappear (oversmoothing) or you see blockiness  
- **Tradeoff:** Larger `a` ⇒ stronger low-rank signal but more smoothing + heavier compute (`P = a²`)

> **Rule:** choose the **smallest** `a` that still captures one full repeat of your texture.

---

### 2) `K` — number of similar patches
- **Start:** 24  
- **Increase (→ 28–32)** if residual is noisy *and* neighbors are truly similar  
- **Decrease (→ 16–20)** if artifacts appear (wrong neighbors) or runtime spikes  
- **Tradeoff:** Higher `K` helps only when neighbors are good; too high can mix in dissimilar patches

> **Tip:** When you raise `K`, consider raising **`search_radius`** a bit to find those extra good neighbors.

---

### 3) `search_radius` (r) — k-NN window half-size
- **Start:** 16  
- **Increase (→ 20–24)** if repeats are spaced farther apart (you’re missing good matches)  
- **Decrease (→ 12)** if runtime is high or neighbor quality drops  
- **Tradeoff:** Larger `r` finds better matches but increases candidate set and time

> **Pairs that work well:**  
> Strong repeats → `r=16–24`, `K=24–32`  
> Natural scenes → `r=12–16`, `K=16–24`

---

### 4) `tau` & `method` — shrinkage strength & style

**Method choice**
- **`svd_soft` / `nnm`**: baseline, reliable and easy to tune  
- **`wnnm`**: better for **mixed patch quality** (varied brightness/contrast); preserves dominant structure, reduces mush

**Threshold (`tau`)**
- **Start:** auto → `τ = σ · √(2 · max(P, K))`  
- **Oversmoothing?** Residual shows edges, details fade → **lower τ** by 10–20% or switch to `wnnm` with smaller `wnnm_c`  
- **Residual noise?** Output still grainy → **raise τ** by 10–20% or increase `K`

**WNNM weight (`wnnm_c`)**
- **Start:** 1.0  
- **Lower to 0.6–0.8** for **more detail** (lighter shrink)  
- **Raise to 1.2–1.5** for **stronger cleanup** (heavier shrink)

> **Recipe:** Try `svd_soft` first. If textures look “plasticky” or edges fade, switch to **`wnnm` with `wnnm_c≈0.7–0.9`** and keep auto-τ.

---

### 5) `anchor_step` — sampling stride
- **Start:** 4 for final quality  
- **Tune:** Use **8** while exploring (much faster), then return to **4** for the final pass  
- **Effect:** Large impact on speed, mild effect on quality (overlap aggregation still helps)

---

### 6) `mean_normalize`
- **Use `True`** for almost all textured/periodic scenes → better neighbor selection and cleaner low-rank grouping  
- **Consider `False`** only if your image already has uniform mean and you observe tiny contrast shifts (rare)

---

### 7) `kaiser_beta`
- **Start:** 2.0 (range 1.5–3.0)  
- **Lower** if you see slight blur over overlaps  
- **Raise** if you see seams/tiling artifacts

---

### 8) `sigma` — noise level
- **Default:** estimator is fine for i.i.d. Gaussian  
- **Set manually** if noise isn’t Gaussian or the estimator is biased by structure  
- **Impact:** Higher `σ` ⇒ higher auto-τ ⇒ stronger denoise; if you set `σ` manually, re-check τ

---

## Interactions (adjust these together)

- **(K, r):** Lift both together to improve neighbor quality; drop both if runtime/mismatches rise  
- **(τ, method):** If lowering τ still blurs detail, switch **method → `wnnm`** rather than micro-tweaking τ  
- **(a, τ):** Larger patches often need **slightly lower τ** (singular values are larger)  
- **(a, K):** If you increase `a`, you can often **reduce K** a bit with minimal loss

---

## Symptom → Fix quick guide

- **Edges faint / textures “melt”** → ↓ τ by 10–20% **or** `method='wnnm'` with `wnnm_c=0.7–0.9`; possibly ↓ `a` by 1  
- **Output still grainy** → ↑ τ by 10–20%, or ↑ `K` (+4–8), or ↑ `r` (+4)  
- **Weird artifacts / ghosting** → ↓ `K` (−4–8) or ↓ `r` (−4); keep `mean_normalize=True`  
- **Seams / blockiness** → ↑ `kaiser_beta` to 2.5–3.0; ensure `anchor_step ≤ 4`; consider ↑ `K`  
- **Too slow** → ↑ `anchor_step` (use 8 while tuning), ↓ `r`, ↓ `K`; revert to higher-quality settings for the final run

---

## Ready-to-use recipes

**A) Strong periodic texture + heavy noise (typical for your case)**  
- `a = 8–10`, `K = 24–32`, `r = 16–24`, `method = 'wnnm'`, `wnnm_c = 0.8–1.0`,  
  `anchor_step = 4`, `mean_normalize = True`, `kaiser_beta = 2.0–2.5`  
- Start with **auto-τ**; if edges soften, drop `wnnm_c` to ~0.7

**B) Natural imagery (edges + fine detail)**  
- `a = 6–8`, `K = 16–24`, `r = 12–16`, `method = 'svd_soft'`,  
  `anchor_step = 4–6`, `mean_normalize = True`, `kaiser_beta = 2.0`  
- Start with auto-τ, then nudge ±10%

---

## Fast, repeatable tuning routine (≈5–8 minutes)

1. **Speed mode:** `anchor_step=8`, `a=8`, `K=20`, `r=16`, `method='svd_soft'`, auto-τ  
2. Residual shows structure? → **lower τ** by 10–15%  
3. Still noisy? → **raise K to 24–28**  
4. Detail suffers? → switch to **`wnnm`**, `wnnm_c=0.8` (keep current τ)  
5. Need better neighbors? → **raise `r` to 20–24**  
6. Lock values → set **`anchor_step=4`** → run full image

---

## Optional extras (use only if needed)

- **Distance metric (NCC):** If lighting varies, use normalized cross-correlation for k-NN instead of L2  
- **Outlier trim:** After K selection, drop top 10–20% farthest patches before SVD  
- **Local τ:** Scale τ by local variance (stronger in flats, lighter on edges)  
- **Tiling:** For huge images, process overlapping tiles (32–64 px overlap) using the same params

---
